In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import seaborn as sns
import matplotlib.pyplot as plt
import nltk
import spacy
import string
import transformers
from transformers import AutoTokenizer
from itertools import repeat
from collections import defaultdict
from wordcloud import WordCloud, STOPWORDS
import re
from functools import partial
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy import sparse
import xgboost
from sklearn import preprocessing
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelBinarizer
from scipy.sparse import csr_matrix
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedShuffleSplit
import xgboost as xgb
import gc
from sklearn.model_selection import GridSearchCV


sw = set(STOPWORDS)


def get_label(val, typ):
    ret = f"Non-{typ}"
    if val > 2.5: ret = typ
    return(ret)


def show_wordcloud(data, title = None):
    wordcloud = WordCloud(
        background_color='white',
        stopwords=sw,
        max_words=200,
        max_font_size=30, 
        scale=3,
        random_state=1 # chosen at random by flipping a coin; it was heads
    ).generate(str(data))

    fig = plt.figure(1, figsize=(12, 12))
    plt.axis('off')
    if title: 
        fig.suptitle(title, fontsize=20)
        fig.subplots_adjust(top=2.3)
    plt.imshow(wordcloud)
    plt.show()
    
def remove_newline(x):
    x = re.sub(r"\n\n", "", x)
    x = re.sub(r"\r\n\r\n", "", x)
    return(x)

from textblob import TextBlob
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_log_error
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# About the Data
In the training files, 8 columns (variables) are provided, out of which there are six target columns. The only input column is the `full_text` which needs to be converted into various features and then analyzed. There are total of 3911 observations in the dataset.

## Look at some of the articles to understand the data.

Following things are observed.
- Newline characters appear. We are removing these.
- There are no emojis, abbreviations, so cleansing may not be required as in normal nlp tasks.

To start with, two new features are created which are called as:
- length of the article
- paragraphs in the article

In [ ]:
df_train = pd.read_csv('/kaggle/input/feedback-prize-english-language-learning/train.csv')
sent  = df_train['full_text'][0]
sent

In [ ]:
df_train['full_text'] = list(map(remove_newline, df_train['full_text'])) 
sent  = df_train['full_text'][1]
sent

**Newline characters are no more present.**

In [ ]:
df_train['length'] = list(map(lambda x : len(x), df_train['full_text']))
df_train['paragraphs'] = list(map(lambda x : len(x.splitlines()), df_train['full_text']))
df_test = pd.read_csv('/kaggle/input/feedback-prize-english-language-learning/test.csv')
df_test['length'] = list(map(lambda x : len(x), df_test['full_text']))
df_test['paragraphs'] = list(map(lambda x : len(x.splitlines()), df_test['full_text']))
df_train['label_cohesion'] = list(map(get_label, df_train['cohesion'],repeat("Cohesive")))
df_train['label_syntax'] = list(map(get_label, df_train['syntax'],repeat("Syntactic")))
df_train['label_vocabulary'] = list(map(get_label, df_train['vocabulary'],repeat("Vocab")))
df_train['label_phraseology'] = list(map(get_label, df_train['phraseology'],repeat("Phrase")))
df_train['label_grammar'] = list(map(get_label, df_train['grammar'],repeat("Grammar")))
df_train['label_conventions'] = list(map(get_label, df_train['conventions'],repeat("Conventional")))

## Examining distribution of Various output columns

It appears that positive classes for all target variables are higher than the negative classes. We expect to see more labels with a score in excess of 2.5

In [ ]:
labels = ['label_cohesion', 'label_syntax', 'label_vocabulary',
       'label_phraseology', 'label_grammar', 'label_conventions']
n = len(labels)
fig,ax = plt.subplots(1,n, figsize=(n*2+ 2,4), sharey=True)
for i, col in enumerate(labels):
    plt.sca(ax[i])
    a = sns.countplot(x= col, data = df_train)
    a.set(xlabel=None)

## Impact of length  and paragraph on output

This is examined as below. We see that the length of article differ slightly where the articles are scored higher on various parameters, so this variable may not be a good variable for prediction purposes.

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize = (15,4))
sns.boxplot(data=df_train, x="label_cohesion", y="paragraphs", ax=axs[0] )
sns.boxplot(data=df_train, x="label_cohesion", y="length", ax=axs[1] )
plt.suptitle("Examining Cohesive articles")
axs[0].set(xlabel=None)
axs[1].set(xlabel=None)
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize = (15,4))
sns.boxplot(data=df_train, x="label_syntax", y="paragraphs", ax=axs[0] )
sns.boxplot(data=df_train, x="label_syntax", y="length", ax=axs[1] )
plt.suptitle("Examining Syntactic articles")
axs[0].set(xlabel=None)
axs[1].set(xlabel=None)
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize = (15,4))
sns.boxplot(data=df_train, x="label_vocabulary", y="paragraphs", ax=axs[0] )
sns.boxplot(data=df_train, x="label_vocabulary", y="length", ax=axs[1] )
plt.suptitle("Examining Vocabulary related data")
axs[0].set(xlabel=None)
axs[1].set(xlabel=None)
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize = (15,4))
sns.boxplot(data=df_train, x="label_phraseology", y="paragraphs", ax=axs[0] )
sns.boxplot(data=df_train, x="label_phraseology", y="length", ax=axs[1] )
plt.suptitle("Examining Phraseology related data")
axs[0].set(xlabel=None)
axs[1].set(xlabel=None)
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize = (15,4))
sns.boxplot(data=df_train, x="label_grammar", y="paragraphs", ax=axs[0] )
sns.boxplot(data=df_train, x="label_grammar", y="length", ax=axs[1] )
plt.suptitle("Examining Grammar related data")
axs[0].set(xlabel=None)
axs[1].set(xlabel=None)
plt.show()

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize = (15,4))
sns.boxplot(data=df_train, x="label_conventions", y="paragraphs", ax=axs[0] )
sns.boxplot(data=df_train, x="label_conventions", y="length", ax=axs[1] )
plt.suptitle("Examining Convention related data")
axs[0].set(xlabel=None)
axs[1].set(xlabel=None)
plt.show()

## Are good articles appropriately puntucated?

Looks like they are. We observe that those articles who score higher on Grammar, Syntax and Conventions, are clearly highly punctuated. Perhaps, a better measure would be to do a ratio.

In [ ]:
df_train['punctuation_count'] = df_train['full_text'].apply(lambda x: len([c for c in str(x) if c in string.punctuation]))
g = sns.FacetGrid(df_train, col="label_syntax")
g.map_dataframe(sns.histplot, x="punctuation_count", binwidth=10, binrange=(0, 200));

In [ ]:
g = sns.FacetGrid(df_train, col="label_grammar")
g.map_dataframe(sns.histplot, x="punctuation_count", binwidth=10, binrange=(0, 200));

In [ ]:
g = sns.FacetGrid(df_train, col="label_conventions")
g.map_dataframe(sns.histplot, x="punctuation_count", binwidth=10, binrange=(0, 200));

In [ ]:
COHESIVE_ARTICLES = df_train['label_cohesion'] == "Cohesive"

from wordcloud import STOPWORDS
def generate_ngrams(text, n_gram=1):
    token = [token for token in text.lower().split(' ') if token != '' if token not in STOPWORDS]
    ngrams = zip(*[token[i:] for i in range(n_gram)])
    return [' '.join(ngram) for ngram in ngrams]



cohesive_unigrams = defaultdict(int)
non_cohesive_unigrams = defaultdict(int)

for sent in df_train[COHESIVE_ARTICLES]['full_text']:
    for word in generate_ngrams(sent):
        cohesive_unigrams[word] += 1
        
for sent in df_train[~COHESIVE_ARTICLES]['full_text']:
    for word in generate_ngrams(sent):
        non_cohesive_unigrams[word] += 1

## Unigrams
A lot of unigrams appear to be common on cohesive and non-cohesive articles.

In [ ]:
df_cohesive_unigrams = pd.DataFrame(sorted(cohesive_unigrams.items(), key=lambda x: x[1])[::-1])
df_non_cohesive_unigrams = pd.DataFrame(sorted(non_cohesive_unigrams.items(), key=lambda x: x[1])[::-1])
N = 25
fig, axes = plt.subplots(ncols=2, figsize=(18, 20), dpi=100)
plt.tight_layout()

sns.barplot(y=df_non_cohesive_unigrams[0].values[:N], x=df_non_cohesive_unigrams[1].values[:N], ax=axes[0], color='red')
sns.barplot(y=df_cohesive_unigrams[0].values[:N], x=df_cohesive_unigrams[1].values[:N], ax=axes[1], color='green')

for i in range(2):
    axes[i].spines['right'].set_visible(False)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')
    axes[i].tick_params(axis='x', labelsize=13)
    axes[i].tick_params(axis='y', labelsize=13)

axes[0].set_title(f'Top {N} most common unigrams in Non-Cohesive articles', fontsize=15)
axes[1].set_title(f'Top {N} most common unigrams in Cohesive articles', fontsize=15)

plt.show()

## Unigram Observation

Observing Unigrams tells us that there are words with single frequency. A feature containing count of unigram words appearing once would be useful here. 

In [ ]:
df_cohesive_unigrams = pd.DataFrame(sorted(cohesive_unigrams.items(), key=lambda x: x[1], reverse = True)[::-1])
df_non_cohesive_unigrams = pd.DataFrame(sorted(non_cohesive_unigrams.items(), key=lambda x: x[1], reverse = True)[::-1])
N = 25
fig, axes = plt.subplots(ncols=2, figsize=(18, 20), dpi=100)
plt.tight_layout()

sns.barplot(y=df_non_cohesive_unigrams[0].values[:N], x=df_non_cohesive_unigrams[1].values[:N], ax=axes[0], color='red')
sns.barplot(y=df_cohesive_unigrams[0].values[:N], x=df_cohesive_unigrams[1].values[:N], ax=axes[1], color='green')

for i in range(2):
    axes[i].spines['right'].set_visible(False)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')
    axes[i].tick_params(axis='x', labelsize=13)
    axes[i].tick_params(axis='y', labelsize=13)

axes[0].set_title(f'Top {N} most common unigrams in Non-Cohesive articles', fontsize=15)
axes[1].set_title(f'Top {N} most common unigrams in Cohesive articles', fontsize=15)

plt.show()

## Creating a list of words which appear upto 3 times in the corpus

These are likely to be spelling mistakes so a count of these may become helpful when building a model. 

In [ ]:
unigrams = defaultdict(int)
for i, sent in enumerate(df_train['full_text']):
    for word in generate_ngrams(sent):
        unigrams[word] += 1
lst_single_occurence = [k for k,v in unigrams.items() if (v < 4)]
print("Total Single Occurence words found:", len(lst_single_occurence))
def count_single_occurence_words(text):
    #text = df_train.iloc[0]['full_text']
    tokens = [token for token in text.lower().split(' ') if token != '' if token not in STOPWORDS]
    single_tokens = [token for token in tokens if token in lst_single_occurence]
    return(len(single_tokens))

df_train['sng_occur_count'] = list(map(count_single_occurence_words, df_train['full_text']))

In [ ]:
g = sns.FacetGrid(df_train, col="label_conventions")
g.map_dataframe(sns.histplot, x="sng_occur_count", kde = True, binwidth=10, binrange=(0, 200));

In [ ]:
g = sns.FacetGrid(df_train, col="label_grammar")
g.map_dataframe(sns.histplot, x="sng_occur_count", kde = True, binwidth=10, binrange=(0, 200));

In [ ]:
g = sns.FacetGrid(df_train, col="label_cohesion")
g.map_dataframe(sns.histplot, x="sng_occur_count", kde = True, binwidth=10, binrange=(0, 200));

# Looking at Wordcloud

Having a look at the Wordcloud of the whole corpus. Sometimes, this also gives useful information.

In [ ]:
show_wordcloud(df_train['full_text'], "WordCloud of the Corpus" )

# SpellCheck

As we see that there are some spelling mistakes seen in the wordcloud for example, "Distance" is spelled as "Distence". It could be useful if we can detect spelling mistakes. 

**Note:** The implementation of SpellCheck takes a lot of time to run, hence this is currently commented out. For the interested people, this is found in the following code snippet. (Show Hidden Code).

In [ ]:
def spell_mistake(word):
    ret = False
    x = TextBlob(word)
    if str(x.correct()) != word:
        ret = True
    return(ret)

def count_of_spell_mistakes(text):
    tokens = [token for token in text.lower().split(' ') if token != '' if token not in STOPWORDS]
    miss_spelt_tokens = [token for token in tokens if spell_mistake(token) ]
    return(len(miss_spelt_tokens))

# df_train['spell_mistake_count'] = list(map(count_of_spell_mistakes, df_train['full_text']))
# g = sns.FacetGrid(df_train, col="label_cohesion")
# g.map_dataframe(sns.histplot, x="spell_mistake_count", kde = True, binwidth=10, binrange=(0, 200));

# TFIDF Embeddings and T-SNE Visualization


In [ ]:
tf_idf_vectorizer = TfidfVectorizer(analyzer="word", use_idf=True, smooth_idf=True, ngram_range=(2, 3))
tf_idf_matrix = tf_idf_vectorizer.fit_transform(df_train['full_text'])
# t-SNE plot
X = tf_idf_matrix.todense()
embeddings = TSNE(n_components=2)
Y = embeddings.fit_transform(X)
plt.scatter(Y[:, 0], Y[:, 1], cmap=plt.cm.Spectral)
plt.show()

In [ ]:
def tokenizer(text):
    if text:
        result = re.findall('[a-z]{2,}', text.lower())
    else:
        result = []
    return result

# Build tfidf vectorizer based model using SGD Regressor

In [ ]:
X = df_train['full_text'].values
y = df_train['cohesion'].values
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.3, random_state=273)
vect = TfidfVectorizer(tokenizer=tokenizer, stop_words='english')
start = time.time()
X_train_vect = vect.fit_transform(X_train)
end = time.time()
print('Time to train vectorizer and transform training text: %0.2fs' % (end - start))

model = SGDRegressor(loss='squared_loss', penalty='l2', random_state=273, max_iter=5)
params = {'penalty':['none','l2','l1'],
          'alpha':[1e-4, 2e-4, 5e-4, 1e-3, 2e-3, 5e-3, 1e-2, 2e-2, 5e-2, 0.1]}
gs = GridSearchCV(estimator=model,
                  param_grid=params,
                  scoring='neg_mean_squared_error',
                  n_jobs=1,
                  cv=5,
                  verbose=3)
start = time.time()
gs.fit(X_train_vect, y_train)
end = time.time()
print('Time to train model: %0.2fs' % (end -start))

In [ ]:
model = gs.best_estimator_
print(gs.best_params_)
print(gs.best_score_)
pipe = Pipeline([('vect',vect),('model',model)])
start = time.time()
y_pred = pipe.predict(X_test)
end = time.time()
print('Time to generate predictions on test set: %0.2fs' % (end - start))
#y_pred
# y_test

# XGBOOST Regression with GridSearchCV

In [ ]:
def get_feature_label(data, label):
    #data_after = data[(data['price']<400) & (data['price']>1)]
    train_features = data[['full_text', 'length', 'paragraphs', 'punctuation_count', 'pc_ratio']]
    ### log transform
    train_labels =  data[label]
    #train_labels[train_labels==0]=0.01
    train_labels = np.log(train_labels)
    return train_features,train_labels


def get_train_test_features(label, df_train, df_test):
    nrow_train = df_train.shape[0]
    train_features,train_labels = get_feature_label(df_train,label)
    df_combine:pd.DataFrame = pd.concat([train_features,df_test[['full_text', 'length', 'paragraphs']]])
    del df_train
    del df_test
    del train_features
    tfidf = TfidfVectorizer(norm='l2',sublinear_tf=True,ngram_range=(1,3),min_df=10,max_features=500, stop_words = 'english')
    X_full_text  = tfidf.fit_transform(df_combine['full_text'])
    del(tfidf)
    X_full_text = X_full_text[:, np.array(np.clip(X_full_text.getnnz(axis=0) - 1, 0, 1), dtype=bool)]
    print ('Dimension of fulltext_features'+str(X_full_text.shape))
    gc.collect()
    X_numerical = df_combine[['length', 'paragraphs']]
    final_features = sparse.hstack((X_full_text, X_numerical)).tocsr()
    print ('Dimension of final_features'+ str(final_features.shape))
    train_final_features = final_features[:nrow_train]
    test_final_features = final_features[nrow_train:]
    del final_features
    gc.collect()
    X = (train_final_features)
    y = (train_labels)
    return(X,y, test_final_features)

In [ ]:
def search_best_params(X,y, label):
    regressor=xgb.XGBRegressor(eval_metric='rmsle',
                           booster='gbtree',
                           objective = 'reg:linear',
                           gamma=0,subsample=0.9,
                           colsample_bytree=1,
                           min_child_weight=1, 
                           n_jobs=4,
                           seed=273
                          )
    param_grid = {"max_depth":    [4, 5, 7, 9, 11],
              "n_estimators": [500, 600, 700, 1000, 1500, 2000],
              "learning_rate": [0.01, 0.015, .1, .25, .5]
             }
    
    param_grid = {"max_depth":    [5],
              "n_estimators": [500],
              "learning_rate": [0.01]
             }
    search = GridSearchCV(regressor, param_grid, cv=5).fit(X, y)
    print(f"The best hyperparameters for {label} are ",search.best_params_)
    return(search)

In [ ]:
df_train = pd.read_csv('/kaggle/input/feedback-prize-english-language-learning/train.csv')
df_train['full_text'] = list(map(remove_newline, df_train['full_text'])) 
df_train['length'] = list(map(lambda x : len(x), df_train['full_text']))
df_train['paragraphs'] = list(map(lambda x : len(x.splitlines()), df_train['full_text']))
df_test = pd.read_csv('/kaggle/input/feedback-prize-english-language-learning/test.csv')
df_test['length'] = list(map(lambda x : len(x), df_test['full_text']))
df_test['paragraphs'] = list(map(lambda x : len(x.splitlines()), df_test['full_text']))
df_train['punctuation_count'] = df_train['full_text'].apply(lambda x: len([c for c in str(x) if c in string.punctuation]))
df_test['punctuation_count'] = df_test['full_text'].apply(lambda x: len([c for c in str(x) if c in string.punctuation]))
df_train['pc_ratio'] = df_train['punctuation_count']/ df_train['length']
df_test['pc_ratio'] = df_test['punctuation_count']/ df_test['length']

In [ ]:
def get_final_model(X, y, search):
    regressor=xgb.XGBRegressor(learning_rate = search.best_params_["learning_rate"],
                               n_estimators  = search.best_params_["n_estimators"],
                               max_depth     = search.best_params_["max_depth"],
                               eval_metric='rmsle',
                               booster='gbtree',
                               objective = 'reg:linear',
                               gamma=0,subsample=0.9,
                               colsample_bytree=1,
                               min_child_weight=1, 
                               n_jobs=4,
                               seed=273
                              )
#X_train,X_test,y_train,y_test = train_test_split(X,y, test_size = 0.2)
    regressor.fit(X, y)
    return(regressor)

In [ ]:
label = 'cohesion'
X,y, test_X = get_train_test_features(label, df_train, df_test); 
search = search_best_params(X,y, label)
xgb_model = get_final_model(X, y, search)
predictions = xgb_model.predict(test_X)
cohesion_label = np.exp(predictions)
cohesion_label = [round(l)/2 for l in cohesion_label*2]


In [ ]:
label = 'syntax'
X,y, test_X = get_train_test_features(label, df_train, df_test); 
search = search_best_params(X,y, label)
xgb_model = get_final_model(X, y, search)
predictions = xgb_model.predict(test_X)
syntax_label = np.exp(predictions)
syntax_label = [round(l)/2 for l in syntax_label*2]

In [ ]:
label = 'vocabulary'
X,y, test_X = get_train_test_features(label, df_train, df_test); 
search = search_best_params(X,y, label)
xgb_model = get_final_model(X, y, search)
predictions = xgb_model.predict(test_X)
vocab_label = np.exp(predictions)
vocab_label = [round(l)/2 for l in vocab_label*2]

In [ ]:
label = 'phraseology'
X,y, test_X = get_train_test_features(label, df_train, df_test); 
search = search_best_params(X,y, label)
xgb_model = get_final_model(X, y, search)
predictions = xgb_model.predict(test_X)
phrase_label = np.exp(predictions)
phrase_label = [round(l)/2 for l in phrase_label*2]

In [ ]:
label = 'grammar'
X,y, test_X = get_train_test_features(label, df_train, df_test); 
search = search_best_params(X,y, label)
xgb_model = get_final_model(X, y, search)
predictions = xgb_model.predict(test_X)
grammar_label = np.exp(predictions)
grammar_label = [round(l)/2 for l in grammar_label*2]

In [ ]:
label = 'conventions'
X,y, test_X = get_train_test_features(label, df_train, df_test); 
search = search_best_params(X,y, label)
xgb_model = get_final_model(X, y, search)
predictions = xgb_model.predict(test_X)
conventions_label = np.exp(predictions)
conventions_label = [round(l)/2 for l in conventions_label*2]

In [ ]:
df_submit = pd.DataFrame({ 'text_id': df_test['text_id']
                           ,'cohesion' : cohesion_label
                           , 'syntax' : syntax_label
                           , 'vocabulary' : vocab_label
                           , 'phraseology' : phrase_label
                           , 'grammar' : grammar_label
                           , 'conventions' : conventions_label
                           })
df_submit.to_csv("submission.csv", index = False)

Thats it.
This completes the data exploration and model building for this task. Thanks for reading. Please send your suggestion on what would you like to see. 